# Polylines and streamlines

**Geometry types:** `polyline` · `streamline`

Ordered vertex sequences. Use `streamline` for tractography data (adds
step-size and propagation metadata). All read/write functions are identical
for both types.

1. Generate synthetic tractography streamlines
2. Write with groups and per-object attributes
3. Open lazily and inspect metadata
4. Read all / by object ID / by group
5. Spatial bounding-box query
6. Build multi-resolution pyramid with object sparsity
7. Read at coarser levels
8. Export to TRK


In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import ZarrVectorStore
from zarr_vectors.types.polylines import write_polylines, read_polylines
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "tracts.zarrvectors")
print("Store path:", STORE)


## 1 · Generate synthetic tractography streamlines

In [ ]:
rng = np.random.default_rng(1)
N = 2_000   # 2000 streamlines

streamlines = []
for i in range(N):
    n_verts = rng.integers(30, 100)
    sl = rng.normal(0, 15, (n_verts, 3)).cumsum(axis=0).astype(np.float32)
    # Centre streamlines around origin in a ~300 mm white-matter volume
    sl += rng.uniform(-100, 100, 3).astype(np.float32)
    streamlines.append(sl)

# Per-streamline attributes
lengths  = np.array([np.sum(np.linalg.norm(np.diff(s, axis=0), axis=1))
                     for s in streamlines], dtype=np.float32)
mean_fa  = rng.uniform(0.2, 0.7, N).astype(np.float32)

# Per-vertex FA (one value per vertex along each streamline)
per_vertex_fa = [rng.uniform(0.1, 0.9, len(s)).astype(np.float32)
                 for s in streamlines]

# Group into 3 bundles
bundle_size = N // 3
groups = {
    "CST":  list(range(bundle_size)),
    "AF":   list(range(bundle_size, 2 * bundle_size)),
    "UF":   list(range(2 * bundle_size, N)),
}

print(f"Generated {N} streamlines")
print(f"Length range: {lengths.min():.1f} – {lengths.max():.1f} mm")
print(f"Groups: { {k: len(v) for k,v in groups.items()} }")


## 2 · Write with groups and attributes

In [ ]:
write_polylines(
    STORE,
    streamlines,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    geometry_type="streamline",
    groups=groups,
    object_attributes={
        "length":  lengths,
        "mean_fa": mean_fa,
    },
    attributes={"fa": per_vertex_fa},
    streamline_metadata={
        "step_size":             0.5,
        "step_size_unit":        "mm",
        "propagation_algorithm": "probabilistic",
        "seeding_strategy":      "wm_mask",
    },
)
print("Write complete.")


## 3 · Open lazily and inspect metadata

In [ ]:
store = ZarrVectorStore(STORE)
print(f"geometry_type  : {store.geometry_type}")
print(f"spatial_dims   : {store.spatial_dims}")
print(f"chunk_shape    : {store.chunk_shape}")
print(f"n_objects      : {store.n_objects:,}   (= {N} streamlines)")
print(f"vertex_count   : {store.vertex_count(0):,}")
print(f"bounding_box   :\n  lo={store.bounding_box[0].round(1)}"
      f"\n  hi={store.bounding_box[1].round(1)}")


## 4a · Read all streamlines

In [ ]:
result_all = read_polylines(STORE)
print(f"polyline_count : {result_all['polyline_count']:,}")
print(f"Type of result['polylines']: {type(result_all['polylines'])}")
print(f"Streamline 0 shape: {result_all['polylines'][0].shape}")
print(f"Streamline 1 shape: {result_all['polylines'][1].shape}")
print(f"\nPer-vertex FA example (streamline 0): {result_all['attributes']['fa'][0][:5]}")


## 4b · Read by object ID

In [ ]:
result_id = read_polylines(STORE, object_ids=[0, 42, 999])
print(f"Read {result_id['polyline_count']} streamlines by ID")
print(f"Streamline 42 has {len(result_id['polylines'][1])} vertices")


## 4c · Read by group

In [ ]:
for group_name in ["CST", "AF", "UF"]:
    r = read_polylines(STORE, group_ids=[group_name])
    print(f"Group {group_name}: {r['polyline_count']:,} streamlines")


## 4d · Read per-object attributes without loading geometry

In [ ]:
from zarr_vectors.core.store import open_store

root = open_store(STORE, mode="r")
lengths_stored = root["resolution_0"]["object_attributes"]["length"][:]
fa_stored      = root["resolution_0"]["object_attributes"]["mean_fa"][:]

print(f"mean_fa range: [{fa_stored.min():.3f}, {fa_stored.max():.3f}]")
print(f"length  range: [{lengths_stored.min():.1f}, {lengths_stored.max():.1f}] mm")

# Select streamlines with high FA and length > 50 mm
selected = np.where((fa_stored > 0.55) & (lengths_stored > 50))[0]
print(f"\nHigh-FA long streamlines: {len(selected)}")
result_sel = read_polylines(STORE, object_ids=selected[:10].tolist())
print(f"Loaded {result_sel['polyline_count']} (first 10 of selection)")


## 5 · Spatial bounding-box query

In [ ]:
lo = np.array([-30.0, -30.0, -30.0])
hi = np.array([ 30.0,  30.0,  30.0])

bbox_result = read_polylines(STORE, bbox=(lo, hi))
print(f"Streamlines passing through ±30 mm cube: {bbox_result['polyline_count']:,}")
print("(Full streamline geometry is returned, not clipped to bbox)")


## 6 · Build multi-resolution pyramid with object sparsity

In [ ]:
from zarr_vectors.multiresolution.coarsen import build_pyramid

build_pyramid(
    STORE,
    level_configs=[
        {"bin_ratio": (2, 2, 2), "object_sparsity": 1.0,
         "sparsity_strategy": "spatial_coverage"},
        {"bin_ratio": (4, 4, 4), "object_sparsity": 0.25,
         "sparsity_strategy": "spatial_coverage"},
    ],
    attribute_aggregation={"fa": "mean"},
)
print("Pyramid built.")


## 7 · Read at coarser levels

In [ ]:
for lv in store.levels:
    n_obj  = store.object_count_at(lv)
    n_vert = store.vertex_count(lv)
    bs     = store.bin_shape_at(lv)
    print(f"Level {lv}: {n_obj:>5,} objects  {n_vert:>8,} vertices  bin_shape={bs}")

print()
coarse = read_polylines(STORE, level=2)
print(f"Level 2 average vertices/streamline: "
      f"{sum(len(s) for s in coarse['polylines']) / max(1, len(coarse['polylines'])):.1f}")


## 8 · Validate

In [ ]:
result_v = validate(STORE, level=3)
print(result_v.summary())


## Summary

| Step | API |
|------|-----|
| Write | `write_polylines(path, streamlines, groups=..., object_attributes=..., attributes=...)` |
| Read all | `read_polylines(path)` |
| Read by ID | `read_polylines(path, object_ids=[...])` |
| Read by group | `read_polylines(path, group_ids=["CST"])` |
| Bbox query | `read_polylines(path, bbox=(lo, hi))` |
| Pyramid | `build_pyramid(path, level_configs)` |
